In [1]:
from ultralytics import YOLO

In [2]:
from ultralytics import settings
settings.update({"datasets_dir": "/home/tj_students/leaf-localisation-main"})

In [3]:
model = YOLO("models/best_m.pt")

In [4]:
metrics = model.val(data="../data.yaml", imgsz=1280)
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")
print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")

Ultralytics 8.4.104 🚀 Python-3.10.12 torch-2.13.0+cu130 CUDA:0 (NVIDIA RTX A5000, 24111MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 9511.3±3138.7 MB/s, size: 1692.2 KB)
val: Scanning /home/tj_students/leaf-localisation-main/data/processed/val/labels.cache... 567 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 567/567 158.5Mit/s 0.0s
val: /home/tj_students/leaf-localisation-main/data/processed/val/images/soycotton_2b47c43d-2fed-4e06-ace5-47c6744417ad.jpeg: 1 duplicate labels removed
val: /home/tj_students/leaf-localisation-main/data/processed/val/images/soycotton_38285a73-29ba-4efd-bf1b-73b6ad7eaf94.jpeg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 16% ━━────────── 6/36 2.4it/s 2.7s<12.6s

[W805 11:51:45.580386810 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1247805440 bytes (free: 1167065088, total: 25282478080).


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 22% ━━╸───────── 8/36 2.6it/s 3.4s<10.7s

[W805 11:51:46.327456552 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1335885824 bytes (free: 647102464, total: 25282478080).


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 58% ━━━━━━━───── 21/36 2.9it/s 8.0s<5.2s

[W805 11:51:51.914278072 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1549795328 bytes (free: 1433141248, total: 25282478080).


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 63% ━━━━━━━╸──── 23/36 2.6it/s 8.9s<5.0s

[W805 11:51:52.800558334 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1763704832 bytes (free: 774569984, total: 25282478080).


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.6it/s 13.8s0.4s
                   all        567       4129      0.803      0.772      0.854      0.688
Speed: 2.0ms preprocess, 18.7ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /home/tj_students/leaf-localisation-main/deployment/runs/detect/val-13
Precision  : 0.803
Recall     : 0.772
mAP50      : 0.854
mAP50-95   : 0.688


In [ ]:
model.export(format="litert", quantize="int8", data="../data.yaml", imgsz=1280)

In [ ]:
model.export(format="tflite", int8=True, data="../data.yaml", imgsz=1280)

In [ ]:
int8_metrics = YOLO("<paste path from Cell 5>").val(data="../data.yaml", imgsz=1280)
print(f"Precision  : {int8_metrics.box.mp:.3f}")
print(f"Recall     : {int8_metrics.box.mr:.3f}")
print(f"mAP50      : {int8_metrics.box.map50:.3f}")
print(f"mAP50-95   : {int8_metrics.box.map:.3f}")

In [ ]:
from pathlib import Path
Path("compiler_logs").mkdir(parents=True, exist_ok=True)

In [ ]:
!edgetpu_compiler "<paste same path from Cell 5>" | tee compiler_logs/edgetpu_log.txt

In [ ]:
model.export(format="ncnn", imgsz=1280)

In [ ]:
ncnn_metrics = YOLO("<paste path from Cell 8, e.g. models/best_m_ncnn_model>").val(data="../data.yaml", imgsz=1280)
print(f"Precision  : {ncnn_metrics.box.mp:.3f}")
print(f"Recall     : {ncnn_metrics.box.mr:.3f}")
print(f"mAP50      : {ncnn_metrics.box.map50:.3f}")
print(f"mAP50-95   : {ncnn_metrics.box.map:.3f}")

In [ ]:
import pandas as pd
from pathlib import Path

rows = []
for name, m in [("pt (baseline)", metrics), ("int8 tflite", int8_metrics), ("ncnn", ncnn_metrics)]:
    rows.append({"format": name, "precision": m.box.mp, "recall": m.box.mr,
                  "mAP50": m.box.map50, "mAP50-95": m.box.map})
df = pd.DataFrame(rows)
Path("reports").mkdir(parents=True, exist_ok=True)
df.to_csv("reports/deployment_metrics.csv", index=False)
df